# E7 - YOLO26s + SCConv + train-only hard-example weighted sampling

This notebook keeps the E5 architecture exactly: two `C3k2_SCConv` replacements at P3/8 and P4/16. The only E7 change is train-split hard-example mining followed by capped `WeightedRandomSampler` sampling.

The E5 checkpoint is used for train-image mining and the explicitly requested same-validation comparison. It is never used to initialize E7; E7 is rebuilt from a fresh `yolo26s.pt` instance. No test split is loaded or evaluated.

In [ ]:
%pip install -q ultralytics==8.4.155
%pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
from google.colab import drive
import inspect
import math
import os
import random
import shutil
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics
import yaml
from PIL import Image
from torch.utils.data import WeightedRandomSampler
from ultralytics import YOLO
from ultralytics.data import build_dataloader
from ultralytics.data.build import InfiniteDataLoader, seed_worker
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import RANK
from ultralytics.utils.torch_utils import get_torch_device_backend, torch_distributed_zero_first

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/DetectedHelmets/DataSet2', exist_ok=True)
os.chdir('/content/DetectedHelmets')
data_root = Path('/content/DetectedHelmets/DataSet2')
drive_dataset_path = Path('/content/drive/MyDrive/DataSet2')
if drive_dataset_path.exists():
    if data_root.is_symlink():
        data_root.unlink()
    elif data_root.exists() and data_root.resolve() != drive_dataset_path.resolve():
        shutil.rmtree(data_root)
    if not data_root.exists():
        data_root.symlink_to(drive_dataset_path, target_is_directory=True)
else:
    import gdown
    archive_path = Path('/content/DetectedHelmets/dataset.rar')
    file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(archive_path), quiet=False, fuzzy=True)
    os.system(f'unrar x -o+ {archive_path} {data_root}/')

DATA_YAML_PATH = Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
MODEL_NAME = 'yolo26s.pt'
ULTRALYTICS_VERSION = '8.4.155'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_scconv_hardmining_dataset2'
E5_EXPERIMENT_NAME = 'helmet_yolo26s_scconv_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
E5_BEST_MODEL_PATH = OUTPUT_DIR / E5_EXPERIMENT_NAME / 'weights' / 'best.pt'
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
HARD_REPORT_PATH = Path('/content/DetectedHelmets/hard_samples_train.csv')
VALIDATION_SPLIT = 'val'
MINING_SPLIT = 'train'
TEST_USED = False
TRAIN_CONFIG = {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15,
    'pretrained': True,
}

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'The E7 protocol requires device=0 and an enabled GPU.'
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert DATA_YAML_PATH.exists(), DATA_YAML_PATH
assert not EXPERIMENT_DIR.exists(), f'Output already exists; choose a fresh run directory: {EXPERIMENT_DIR}'
with DATA_YAML_PATH.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)
raw_names = data_config.get('names', [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
CLASS_NAMES = list(raw_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, CLASS_NAMES

yaml_root = DATA_YAML_PATH.parent.resolve()
configured_root = Path(data_config.get('path', '.'))
if configured_root.is_absolute():
    DATA_ROOT_CANDIDATES = [configured_root.resolve(), yaml_root]
else:
    DATA_ROOT_CANDIDATES = [
        (yaml_root / configured_root).resolve(),
        yaml_root,
    ]
DATA_ROOT_CANDIDATES = list(dict.fromkeys(DATA_ROOT_CANDIDATES))
DATASET_ROOT = next((root for root in DATA_ROOT_CANDIDATES if root.exists()), DATA_ROOT_CANDIDATES[0])
def resolve_split_path(value):
    path = Path(value)
    if path.is_absolute():
        candidates = [path.resolve()]
    else:
        candidates = [(root / path).resolve() for root in DATA_ROOT_CANDIDATES]
        stripped_parts = [part for part in path.parts if part not in {'.', '..'}]
        if stripped_parts:
            candidates.append((yaml_root / Path(*stripped_parts)).resolve())
    candidates = list(dict.fromkeys(candidates))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f'Could not resolve split path {value!r}. Tried: {candidates}'
    )
TRAIN_IMAGE_DIR = resolve_split_path(data_config['train'])
TRAIN_LABEL_DIR = TRAIN_IMAGE_DIR.parent / 'labels'
print('Dataset:', DATA_YAML_PATH)
print('Train images:', TRAIN_IMAGE_DIR)
print('Train labels:', TRAIN_LABEL_DIR)
print('Classes:', CLASS_NAMES)
print('Protocol:', TRAIN_CONFIG)
print('DetectionTrainer.get_dataloader:', inspect.signature(DetectionTrainer.get_dataloader))
assert {'dataset_path', 'batch_size', 'rank', 'mode'}.issubset(inspect.signature(DetectionTrainer.get_dataloader).parameters)

## 1. Discover the E5 graph and reuse its SCConv implementation

The target modules are discovered from the runtime graph rather than hard-coded layer indices. The two selected detection inputs must be the P3/8 and P4/16 `C3k2` blocks.

In [ ]:
base_model = YOLO(MODEL_NAME)

def class_count(module, class_name):
    return sum(1 for item in module.modules() if item.__class__.__name__ == class_name)

def probe_c3k2(yolo_model, imgsz=640):
    records, handles = [], []
    def make_hook(index, module):
        def hook(_module, inputs, output):
            if not torch.is_tensor(output):
                return
            input_value = inputs[0] if inputs else None
            records.append({
                'index': index, 'from': getattr(module, 'f', None),
                'input_shape': tuple(input_value.shape) if torch.is_tensor(input_value) else None,
                'output_shape': tuple(output.shape), 'stride': imgsz // output.shape[-1],
                'bottleneck_count': class_count(module, 'Bottleneck'),
            })
        return hook
    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'C3k2':
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()
    return sorted(records, key=lambda item: item['index'])

c3k2_probe = probe_c3k2(base_model)
detect_entries = [(i, m) for i, m in enumerate(base_model.model.model) if m.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}]
assert detect_entries, 'Detect head not found.'
detect_index, detect_head = min(detect_entries, key=lambda item: item[0])
detect_from = getattr(detect_head, 'f', None)
assert isinstance(detect_from, (list, tuple)), 'Detect input indices unavailable.'
detect_source_indices = {int(index) for index in detect_from}
upsample_indices = [i for i, m in enumerate(base_model.model.model) if m.__class__.__name__ == 'Upsample' and i < detect_index]
assert upsample_indices, 'Neck start could not be found.'
neck_start_index = min(upsample_indices)
neck_candidates = [r for r in c3k2_probe if neck_start_index < r['index'] < detect_index and r['index'] in detect_source_indices and r['stride'] in (8, 16)]
selected_by_stride = {}
for stride in (8, 16):
    candidates = [r for r in neck_candidates if r['stride'] == stride]
    assert len(candidates) == 1, f'Expected one C3k2 Detect input for stride {stride}, got {candidates}'
    selected_by_stride[stride] = candidates[0]
c3k2_specs = [{'level': f'P{stride // 8 + 2}/{stride}', **selected_by_stride[stride]} for stride in (8, 16)]
print('Selected runtime C3k2 targets:', c3k2_specs)
assert {spec['stride'] for spec in c3k2_specs} == {8, 16}

In [ ]:
def nearest_valid_group_num(c_num, requested):
    valid = [group for group in range(1, c_num + 1) if c_num % group == 0]
    return min(valid, key=lambda group: (abs(group - requested), -group))

class GroupBatchnorm2d(nn.Module):
    def __init__(self, c_num, group_num=4, eps=1e-5):
        super().__init__()
        self.group_num = nearest_valid_group_num(c_num, group_num)
        self.weight = nn.Parameter(torch.ones(c_num, 1, 1))
        self.bias = nn.Parameter(torch.zeros(c_num, 1, 1))
        self.eps = eps
    def forward(self, x):
        batch, channels, height, width = x.shape
        grouped = x.reshape(batch, self.group_num, -1)
        mean = grouped.mean(2, keepdim=True)
        variance = grouped.var(2, keepdim=True, unbiased=False)
        normalized = ((grouped - mean) / torch.sqrt(variance + self.eps)).reshape(batch, channels, height, width)
        return normalized * self.weight + self.bias

class SRU(nn.Module):
    def __init__(self, oup_channels, group_num=4, gate_threshold=0.5):
        super().__init__()
        if oup_channels % 2:
            raise ValueError('SRU cross reconstruction requires even channels.')
        self.gn = GroupBatchnorm2d(oup_channels, group_num)
        self.gate_threshold = gate_threshold
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        gn_x = self.gn(x)
        gamma = self.gn.weight
        gamma_weight = (gamma / (gamma.sum() + torch.finfo(gamma.dtype).eps)).view(1, -1, 1, 1)
        weights = self.sigmoid(gn_x * gamma_weight)
        informative = torch.where(weights > self.gate_threshold, torch.ones_like(weights), weights)
        non_informative = torch.where(weights > self.gate_threshold, torch.zeros_like(weights), weights)
        return self.reconstruct(informative * x, non_informative * x)
    def reconstruct(self, x1, x2):
        half = x1.shape[1] // 2
        x11, x12 = torch.split(x1, [half, half], 1)
        x21, x22 = torch.split(x2, [half, half], 1)
        return torch.cat([x11 + x22, x12 + x21], 1)

class CRU(nn.Module):
    def __init__(self, op_channel, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__()
        self.up_channel = int(alpha * op_channel)
        self.low_channel = op_channel - self.up_channel
        self.up_squeeze = self.up_channel // squeeze_ratio
        self.low_squeeze = self.low_channel // squeeze_ratio
        if min(self.up_squeeze, self.low_squeeze) < 1 or self.up_squeeze % group_size or op_channel % group_size:
            raise ValueError('CRU channel configuration is incompatible.')
        self.squeeze1 = nn.Conv2d(self.up_channel, self.up_squeeze, 1, bias=False)
        self.squeeze2 = nn.Conv2d(self.low_channel, self.low_squeeze, 1, bias=False)
        self.GWC = nn.Conv2d(self.up_squeeze, op_channel, group_kernel_size, 1, group_kernel_size // 2, groups=group_size)
        self.PWC1 = nn.Conv2d(self.up_squeeze, op_channel, 1, bias=False)
        self.PWC2 = nn.Conv2d(self.low_squeeze, op_channel - self.low_squeeze, 1, bias=False)
        self.advavg = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        upper, lower = torch.split(x, [self.up_channel, self.low_channel], 1)
        upper, lower = self.squeeze1(upper), self.squeeze2(lower)
        upper_branch = self.GWC(upper) + self.PWC1(upper)
        lower_branch = torch.cat([self.PWC2(lower), lower], 1)
        fused = torch.cat([upper_branch, lower_branch], 1)
        fused = F.softmax(self.advavg(fused), dim=1) * fused
        first, second = torch.split(fused, fused.shape[1] // 2, 1)
        return first + second

class ScConv(nn.Module):
    def __init__(self, op_channel, group_num=4, gate_threshold=0.5, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__()
        self.SRU = SRU(op_channel, group_num, gate_threshold)
        self.CRU = CRU(op_channel, alpha, squeeze_ratio, group_size, group_kernel_size)
    def forward(self, x):
        return self.CRU(self.SRU(x))

SCONV_CONFIG = {'group_num': 4, 'gate_threshold': 0.5, 'alpha': 0.5, 'squeeze_ratio': 2, 'group_size': 2, 'group_kernel_size': 3}
unit_input = torch.randn(2, 32, 16, 16, requires_grad=True)
unit_module = ScConv(32)
unit_output = unit_module(unit_input)
assert unit_output.shape == unit_input.shape
unit_output.mean().backward()
assert any(parameter.grad is not None for parameter in unit_module.parameters())
print('SCConv unit shape/gradient checks: OK')

In [ ]:
from ultralytics.nn.modules.block import Bottleneck

class Bottleneck_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.cv1 = original.cv1
        self.add = bool(getattr(original, 'add', False))
        raw_cv2 = getattr(original.cv2, 'conv', original.cv2)
        input_channels, output_channels = int(raw_cv2.in_channels), int(raw_cv2.out_channels)
        stride = getattr(raw_cv2, 'stride', 1)
        stride = (stride, stride) if isinstance(stride, int) else tuple(stride)
        self.scconv_applied = stride == (1, 1) and input_channels == output_channels and input_channels % 2 == 0
        self.cv2 = ScConv(input_channels, **SCONV_CONFIG) if self.scconv_applied else original.cv2
    def forward(self, x):
        output = self.cv2(self.cv1(x))
        return x + output if self.add else output

def replace_bottlenecks(module, stats):
    if module.__class__.__name__ == 'Bottleneck':
        replacement = Bottleneck_SCConv(module)
        stats['seen'] += 1
        stats['scconv'] += int(replacement.scconv_applied)
        stats['kept'] += int(not replacement.scconv_applied)
        return replacement
    for name, child in list(module.named_children()):
        replacement = replace_bottlenecks(child, stats)
        if replacement is not child:
            setattr(module, name, replacement)
    return module

class C3k2_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.c, self.cv1, self.cv2 = original.c, original.cv1, original.cv2
        self.stats = {'seen': 0, 'scconv': 0, 'kept': 0}
        self.m = replace_bottlenecks(original.m, self.stats)
    def forward(self, x):
        values = list(self.cv1(x).chunk(2, 1))
        values.extend(block(values[-1]) for block in self.m)
        return self.cv2(torch.cat(values, 1))
    def forward_split(self, x):
        values = list(self.cv1(x).split((self.c, self.c), 1))
        values.extend(block(values[-1]) for block in self.m)
        return self.cv2(torch.cat(values, 1))

def copy_graph_metadata(new, old, index, from_value):
    new.i = getattr(old, 'i', index)
    new.f = getattr(old, 'f', from_value)
    new.type = getattr(old, 'type', new.__class__.__name__)
    new.np = sum(parameter.numel() for parameter in new.parameters())
    return new

def count_scconv(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'ScConv')

def count_c3k2_scconv(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'C3k2_SCConv')

def sum_tensors(value):
    if torch.is_tensor(value):
        return value.float().sum()
    if isinstance(value, (list, tuple)):
        return sum((sum_tensors(item) for item in value), torch.zeros((), device=value[0].device if value and torch.is_tensor(value[0]) else None))
    if isinstance(value, dict):
        return sum((sum_tensors(item) for item in value.values()), torch.zeros(()))
    return torch.zeros(())

def build_fresh_scconv_model():
    model = YOLO(MODEL_NAME)
    replacement_stats = []
    for spec in c3k2_specs:
        old = model.model.model[spec['index']]
        assert old.__class__.__name__ == 'C3k2'
        new = C3k2_SCConv(old)
        replacement_stats.append((spec['level'], new.stats))
        model.model.model[spec['index']] = copy_graph_metadata(new, old, spec['index'], spec['from'])
    assert count_c3k2_scconv(model.model) == 2
    print('Fresh model replacement stats:', replacement_stats)
    print('C3k2_SCConv:', count_c3k2_scconv(model.model), 'ScConv:', count_scconv(model.model))
    return model

e7_model = build_fresh_scconv_model()
e7_device = next(e7_model.model.parameters()).device
e7_model.model.eval()
with torch.no_grad():
    e7_dummy_output = e7_model.model(torch.randn(2, 3, 640, 640, device=e7_device))
assert any(torch.is_tensor(item) for item in (e7_dummy_output if isinstance(e7_dummy_output, (list, tuple)) else [e7_dummy_output]))
e7_model.model.train()
e7_model.model.zero_grad(set_to_none=True)
e7_graph_output = e7_model.model(torch.randn(2, 3, 640, 640, device=e7_device, requires_grad=True))
e7_graph_loss = sum_tensors(e7_graph_output)
assert e7_graph_loss.requires_grad
e7_graph_loss.backward()
scconv_grad_ok = any(parameter.grad is not None for item in e7_model.model.modules() if item.__class__.__name__ == 'ScConv' for parameter in item.parameters())
print('SCConv gradient OK:', scconv_grad_ok)
assert scconv_grad_ok
e7_model.model.eval()
assert e7_model is not None and e7_model.model is not None

## 2. Train-only dataset statistics

The following cells read only `train/images` and `train/labels`. They do not rewrite labels or copy images.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG', '.BMP', '.WEBP'}
NO_HELMET_ID = 2
WITH_HELMET_ID = 1
MOTORCYCLE_ID = 0

def list_images(image_dir):
    return sorted(path for path in Path(image_dir).iterdir() if path.is_file() and path.suffix in IMAGE_EXTENSIONS)

def label_path_for_image(image_path):
    return TRAIN_LABEL_DIR / f'{image_path.stem}.txt'

def read_yolo_labels(label_path):
    rows = []
    if not label_path.exists():
        return rows
    for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f'Invalid YOLO label at {label_path}:{line_number}')
        class_id, x_center, y_center, width, height = fields
        rows.append((int(float(class_id)), float(x_center), float(y_center), float(width), float(height)))
    return rows

def normalized_path(value):
    return str(Path(value).resolve())

TRAIN_IMAGE_PATHS = list_images(TRAIN_IMAGE_DIR)
assert TRAIN_IMAGE_PATHS, f'No train images found in {TRAIN_IMAGE_DIR}'
assert all(normalized_path(path).startswith(normalized_path(TRAIN_IMAGE_DIR)) for path in TRAIN_IMAGE_PATHS)

class_counts = Counter()
nohelmet_areas, nohelmet_widths, nohelmet_heights, nohelmet_aspects = [], [], [], []
missing_label_paths = []
for image_path in TRAIN_IMAGE_PATHS:
    label_path = label_path_for_image(image_path)
    if not label_path.exists():
        missing_label_paths.append(label_path)
        continue
    for class_id, _xc, _yc, width, height in read_yolo_labels(label_path):
        class_counts[class_id] += 1
        if class_id == NO_HELMET_ID:
            nohelmet_areas.append(width * height)
            nohelmet_widths.append(width)
            nohelmet_heights.append(height)
            nohelmet_aspects.append(width / height if height else 0.0)
assert not missing_label_paths, f'Missing train labels: {missing_label_paths[:5]}'

def mean_or_zero(values):
    return float(np.mean(values)) if values else 0.0

train_statistics = pd.DataFrame([
    {'Statistic': 'train images', 'Value': len(TRAIN_IMAGE_PATHS)},
    {'Statistic': 'motorcycle objects', 'Value': class_counts[MOTORCYCLE_ID]},
    {'Statistic': 'rider_with_helmet objects', 'Value': class_counts[WITH_HELMET_ID]},
    {'Statistic': 'rider_without_helmet objects', 'Value': class_counts[NO_HELMET_ID]},
    {'Statistic': 'no-helmet bbox area mean', 'Value': mean_or_zero(nohelmet_areas)},
    {'Statistic': 'no-helmet bbox width mean', 'Value': mean_or_zero(nohelmet_widths)},
    {'Statistic': 'no-helmet bbox height mean', 'Value': mean_or_zero(nohelmet_heights)},
    {'Statistic': 'no-helmet bbox aspect ratio mean', 'Value': mean_or_zero(nohelmet_aspects)},
    {'Statistic': 'no-helmet small-object ratio (area < 0.01)', 'Value': float(np.mean(np.asarray(nohelmet_areas) < 0.01)) if nohelmet_areas else 0.0},
    {'Statistic': 'no-helmet medium-object ratio (0.01 <= area < 0.05)', 'Value': float(np.mean((np.asarray(nohelmet_areas) >= 0.01) & (np.asarray(nohelmet_areas) < 0.05))) if nohelmet_areas else 0.0},
    {'Statistic': 'no-helmet large-object ratio (area >= 0.05)', 'Value': float(np.mean(np.asarray(nohelmet_areas) >= 0.05)) if nohelmet_areas else 0.0},
])
display(train_statistics)
assert MINING_SPLIT == 'train' and not any('/val/' in normalized_path(path).replace('\\', '/') for path in TRAIN_IMAGE_PATHS)

## 3. Mine hard examples with the E5 checkpoint

Mining uses the E5 best checkpoint only on train images with `imgsz=640`, `conf=0.25`, `iou=0.5`, and `augment=False`. Ground-truth no-helmet boxes are matched at IoU >= 0.5.

In [ ]:
def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def extract_checkpoint_state(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('model', 'ema', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module):
                return value.state_dict()
            if isinstance(value, dict):
                tensors = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensors:
                    return tensors
        tensors = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensors:
            return tensors
    raise TypeError('Checkpoint state dict not found.')

def audit_checkpoint_transfer(model, state, label):
    target = model.state_dict()
    compatible = {name: value for name, value in state.items() if name in target and tuple(value.shape) == tuple(target[name].shape)}
    if not compatible:
        raise RuntimeError(f'{label}: no compatible checkpoint tensors.')
    target_count = len(target)
    compatible_count = len(compatible)
    transfer_ratio = compatible_count / target_count if target_count > 0 else 0.0
    print('Checkpoint transfer:', compatible_count, '/', target_count, '=', f'{transfer_ratio:.2%}')
    assert transfer_ratio >= 0.95, (
        f'{label}: checkpoint load is incomplete. Transfer ratio is only {transfer_ratio:.2%}.'
    )
    report = model.load_state_dict(compatible, strict=False)
    print('Missing keys:', len(report.missing_keys))
    print('Unexpected keys:', len(report.unexpected_keys))
    print('First missing keys:', report.missing_keys[:20])
    print('First unexpected keys:', report.unexpected_keys[:20])
    scconv_module_names = [
        name for name, module in model.named_modules()
        if module.__class__.__name__ in {'C3k2_SCConv', 'ScConv', 'SRU', 'CRU'}
    ]
    critical_missing = [
        key for key in report.missing_keys
        if any(name and key.startswith(name + '.') for name in scconv_module_names)
    ]
    print('Critical SCConv missing keys:', critical_missing[:20])
    assert not critical_missing, (
        f'{label}: checkpoint is missing SCConv weights: {critical_missing[:20]}'
    )
    return report

def load_custom_scconv_checkpoint(path):
    state = extract_checkpoint_state(torch_load_checkpoint(str(path)))
    if all(name.startswith('module.') for name in state):
        state = {name[7:]: value for name, value in state.items()}
    try:
        direct = YOLO(str(path))
        if count_c3k2_scconv(direct.model) == 2:
            audit_checkpoint_transfer(direct.model, state, 'Direct checkpoint')
            return direct, 'direct'
    except Exception as direct_error:
        print('Direct custom checkpoint load unavailable:', direct_error)
    rebuilt = build_fresh_scconv_model()
    audit_checkpoint_transfer(rebuilt.model, state, 'Rebuilt checkpoint')
    assert count_c3k2_scconv(rebuilt.model) == 2
    return rebuilt, 'rebuilt'

assert E5_BEST_MODEL_PATH.exists(), E5_BEST_MODEL_PATH
e5_model, e5_load_mode = load_custom_scconv_checkpoint(E5_BEST_MODEL_PATH)
print('E5 checkpoint load mode:', e5_load_mode)
print('C3k2_SCConv IN E5:', count_c3k2_scconv(e5_model.model))
e5_mining_model = e5_model.model
assert count_c3k2_scconv(e5_mining_model) == 2
e5_device = next(e5_mining_model.parameters()).device
dummy = torch.randn(2, 3, 640, 640, device=e5_device)
with torch.no_grad():
    _ = e5_mining_model(dummy)
print('E5 mining model forward: OK')
assert e5_model is not e7_model

In [ ]:
def xywhn_to_xyxy(box, image_width, image_height):
    _xc, _yc, width, height = box
    xc, yc = _xc * image_width, _yc * image_height
    half_width, half_height = width * image_width / 2, height * image_height / 2
    return np.asarray([xc - half_width, yc - half_height, xc + half_width, yc + half_height], dtype=np.float32)

def box_iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return np.zeros(0, dtype=np.float32)
    box = np.asarray(box, dtype=np.float32)
    boxes = np.asarray(boxes, dtype=np.float32)
    left_top = np.maximum(box[:2], boxes[:, :2])
    right_bottom = np.minimum(box[2:], boxes[:, 2:])
    intersection = np.prod(np.maximum(right_bottom - left_top, 0.0), axis=1)
    box_area = max(float(np.prod(np.maximum(box[2:] - box[:2], 0.0))), 0.0)
    boxes_area = np.prod(np.maximum(boxes[:, 2:] - boxes[:, :2], 0.0), axis=1)
    union = box_area + boxes_area - intersection
    return intersection / np.maximum(union, 1e-12)

def mine_one_image(model, image_path):
    with Image.open(image_path) as image:
        image_width, image_height = image.size
    raw_labels = read_yolo_labels(label_path_for_image(image_path))
    gt_nohelmet = [xywhn_to_xyxy(row[1:], image_width, image_height) for row in raw_labels if row[0] == NO_HELMET_ID]
    prediction_results = model.predict(
        source=str(image_path), imgsz=640, conf=0.25, iou=0.5, device=0,
        augment=False, save=False, verbose=False, plots=False,
    )
    prediction = prediction_results[0]
    if prediction.boxes is None or len(prediction.boxes) == 0:
        pred_boxes = np.empty((0, 4), dtype=np.float32)
        pred_classes = np.empty((0,), dtype=np.int64)
        pred_confidences = np.empty((0,), dtype=np.float32)
    else:
        pred_boxes = prediction.boxes.xyxy.detach().cpu().numpy()
        pred_classes = prediction.boxes.cls.detach().cpu().numpy().astype(int)
        pred_confidences = prediction.boxes.conf.detach().cpu().numpy()

    missed, misclassified, low_confidence = 0, 0, 0
    for gt_box in gt_nohelmet:
        ious = box_iou_one_to_many(gt_box, pred_boxes)
        matching_indices = np.flatnonzero(ious >= 0.5)
        if len(matching_indices) == 0:
            missed += 1
            continue
        best_index = matching_indices[np.argmax(ious[matching_indices])]
        if pred_classes[best_index] in (MOTORCYCLE_ID, WITH_HELMET_ID):
            misclassified += 1
        elif pred_confidences[best_index] < 0.50:
            low_confidence += 1

    false_positive = 0
    gt_boxes_array = np.asarray(gt_nohelmet, dtype=np.float32)
    for pred_box, pred_class, confidence in zip(pred_boxes, pred_classes, pred_confidences):
        if pred_class != NO_HELMET_ID or confidence < 0.40:
            continue
        if len(gt_boxes_array) == 0 or not np.any(box_iou_one_to_many(pred_box, gt_boxes_array) >= 0.5):
            false_positive += 1

    hard_score = 3 * missed + 3 * misclassified + low_confidence + false_positive
    calculated_weight = 1.0
    if low_confidence or false_positive:
        calculated_weight = max(calculated_weight, 1.5)
    if missed:
        calculated_weight = max(calculated_weight, 2.5)
    if misclassified:
        calculated_weight = max(calculated_weight, 3.0)
    return {
        'image': normalized_path(image_path),
        'num_nohelmet_gt': len(gt_nohelmet),
        'missed_nohelmet': missed,
        'misclassified_nohelmet': misclassified,
        'low_conf_nohelmet': low_confidence,
        'false_positive_nohelmet': false_positive,
        'hard_score': hard_score,
        'sample_weight': min(calculated_weight, 3.0),
    }

hard_rows = [mine_one_image(e5_model, image_path) for image_path in TRAIN_IMAGE_PATHS]
hard_samples = pd.DataFrame(hard_rows, columns=[
    'image', 'num_nohelmet_gt', 'missed_nohelmet', 'misclassified_nohelmet',
    'low_conf_nohelmet', 'false_positive_nohelmet', 'hard_score', 'sample_weight',
])
hard_samples.to_csv(HARD_REPORT_PATH, index=False)
print('Hard report:', HARD_REPORT_PATH)
display(hard_samples.head())
display(pd.DataFrame([{
    'hard images': int((hard_samples['sample_weight'] > 1.0).sum()),
    'missed': int(hard_samples['missed_nohelmet'].sum()),
    'misclassified': int(hard_samples['misclassified_nohelmet'].sum()),
    'low confidence': int(hard_samples['low_conf_nohelmet'].sum()),
    'false positive': int(hard_samples['false_positive_nohelmet'].sum()),
}]))
assert set(hard_samples.columns) == {'image', 'num_nohelmet_gt', 'missed_nohelmet', 'misclassified_nohelmet', 'low_conf_nohelmet', 'false_positive_nohelmet', 'hard_score', 'sample_weight'}
assert hard_samples['sample_weight'].between(1.0, 3.0).all()
assert hard_samples['image'].map(lambda value: normalized_path(TRAIN_IMAGE_DIR) in value).all()

## 4. Version-8.4.155 train-only weighted sampler

`DetectionTrainer.get_dataloader` in 8.4.155 calls `build_dataset(..., mode, batch_size)` and then `build_dataloader(...)`. This subclass follows that contract. The validation branch calls the original builder; the train branch constructs the same `InfiniteDataLoader` with a replacement `WeightedRandomSampler` whose `num_samples` equals the original train-image count.

In [ ]:
hard_weight_lookup = dict(zip(hard_samples['image'], hard_samples['sample_weight']))

def make_weighted_infinite_loader(dataset, batch_size, workers, sampler, drop_last, device, seed):
    dataset_len = len(dataset)
    batch_size = min(batch_size, dataset_len)
    samples = len(sampler)
    drop_last = drop_last and bool(batch_size) and dataset_len % batch_size != 0
    batches = (samples // batch_size if drop_last else math.ceil(samples / batch_size)) if batch_size else 0
    device_type = getattr(device, 'type', str(device).split(':')[0])
    nd = get_torch_device_backend(device).device_count() if device_type not in {'cpu', 'mps'} else 0
    nw = min(os.cpu_count() // max(nd, 1), workers, 0 if batches <= 1 else batches)
    generator = torch.Generator()
    generator.manual_seed((6148914691236517205 + RANK + seed) % (1 << 64))
    pin_memory = nd > 0
    pin_memory_device = device_type if pin_memory and device_type in {'npu', 'xpu'} else None
    return InfiniteDataLoader(
        dataset=dataset, batch_size=batch_size, shuffle=False, num_workers=nw, sampler=sampler,
        prefetch_factor=4 if nw > 0 else None, pin_memory=pin_memory,
        **({'pin_memory_device': pin_memory_device} if pin_memory_device else {}),
        collate_fn=getattr(dataset, 'collate_fn', None),
        worker_init_fn=seed_worker, generator=generator, drop_last=drop_last,
    )

class HardMiningSCConvTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        return e7_model.model

    def setup_model(self):
        self.model = e7_model.model
        print('C3k2_SCConv INSIDE setup_model:', count_c3k2_scconv(self.model))
        assert count_c3k2_scconv(self.model) == 2
        return None

    def get_dataloader(self, dataset_path: str, batch_size: int = 16, rank: int = 0, mode: str = 'train'):
        assert mode in {'train', 'val'}, f'Mode must be train or val, got {mode}'
        with torch_distributed_zero_first(rank):
            dataset = self.build_dataset(dataset_path, mode, batch_size)
        if mode == 'val':
            validation_loader = build_dataloader(
                dataset, batch=batch_size, workers=self.args.workers * 2, shuffle=False,
                rank=rank, drop_last=False, device=self.device,
            )
            assert not isinstance(getattr(validation_loader, 'sampler', None), WeightedRandomSampler)
            return validation_loader

        dataset_paths = [normalized_path(path) for path in dataset.im_files]
        assert len(dataset_paths) == len(set(dataset_paths)), 'dataset.im_files contains duplicate paths.'
        assert set(dataset_paths) == set(hard_weight_lookup.keys()), (
            'Mining images and training dataset do not match exactly.'
        )
        weights = torch.tensor(
            [float(hard_weight_lookup[path]) for path in dataset_paths],
            dtype=torch.double,
        )
        hard_mask = weights.cpu().numpy() > 1.0
        assert len(weights) == len(dataset) == len(TRAIN_IMAGE_PATHS)
        print('Dataset paths:', len(dataset_paths))
        print('Weight lookup:', len(hard_weight_lookup))
        unique_weights, weight_counts = np.unique(weights.cpu().numpy(), return_counts=True)
        print('Weight distribution:', dict(zip(unique_weights, weight_counts)))
        assert float(weights.min()) >= 1.0
        assert float(weights.max()) <= 3.0
        sampler_generator = torch.Generator().manual_seed(TRAIN_CONFIG['seed'])
        sampler = WeightedRandomSampler(weights=weights, num_samples=len(dataset_paths), replacement=True, generator=sampler_generator)
        assert sampler.replacement and sampler.num_samples == len(dataset)

        if not hasattr(self, '_sampler_diagnostics_printed'):
            self._sampler_diagnostics_printed = True
            print('Total train images:', len(dataset))
            print('Normal images:', int((~hard_mask).sum()))
            print('Hard images:', int(hard_mask.sum()))
            print('Sample weight distribution:', hard_samples['sample_weight'].value_counts().sort_index().to_dict())
            diagnostic_sampler = WeightedRandomSampler(
                weights, num_samples=min(max(20000, len(dataset)), 50000), replacement=True,
                generator=torch.Generator().manual_seed(TRAIN_CONFIG['seed']),
            )
            sampled_indices = list(iter(diagnostic_sampler))[:20]
            print('20 sampled train image paths:')
            for index in sampled_indices:
                index = int(index)
                print(' ', dataset_paths[index], 'weight=', float(weights[index]))
            sampled_counts = np.bincount([int(index) for index in iter(WeightedRandomSampler(
                weights, num_samples=min(max(20000, len(dataset)), 50000), replacement=True,
                generator=torch.Generator().manual_seed(TRAIN_CONFIG['seed'] + 1),
            ))], minlength=len(dataset))
            if hard_mask.any() and (~hard_mask).any():
                hard_per_image = sampled_counts[hard_mask].mean()
                normal_per_image = sampled_counts[~hard_mask].mean()
                print('Mean sampled frequency per hard image:', hard_per_image)
                print('Mean sampled frequency per normal image:', normal_per_image)
                assert hard_per_image > normal_per_image

        loader = make_weighted_infinite_loader(
            dataset=dataset, batch_size=batch_size, workers=self.args.workers, sampler=sampler,
            drop_last=bool(self.args.compile), device=self.device, seed=TRAIN_CONFIG['seed'],
        )
        assert isinstance(loader.sampler, WeightedRandomSampler)
        assert loader.sampler.replacement and loader.sampler.num_samples == len(dataset)
        return loader

print('Weighted sampler trainer ready.')

In [ ]:
TRAIN_OVERRIDES = {
    'model': MODEL_NAME, 'data': str(DATA_YAML_PATH), **TRAIN_CONFIG,
    'project': str(OUTPUT_DIR), 'name': EXPERIMENT_NAME,
    'save': True, 'save_period': 10, 'plots': True, 'exist_ok': False,
}

c3k2_scconv_before_train = count_c3k2_scconv(e7_model.model)
assert c3k2_scconv_before_train == 2
trainer = HardMiningSCConvTrainer(overrides=TRAIN_OVERRIDES, _callbacks=e7_model.callbacks)
trainer.model = e7_model.model
c3k2_scconv_inside_trainer = count_c3k2_scconv(trainer.model)
print('C3k2_SCConv BEFORE TRAIN:', c3k2_scconv_before_train)
print('C3k2_SCConv INSIDE TRAINER:', c3k2_scconv_inside_trainer)
assert c3k2_scconv_inside_trainer == 2

train_results = trainer.train()
assert EXPERIMENT_DIR.exists(), EXPERIMENT_DIR
EXPERIMENT_VERSION_FILE = EXPERIMENT_DIR / 'ultralytics_version.txt'
EXPERIMENT_VERSION_FILE.write_text(
    'ultralytics==' + ultralytics.__version__ + chr(10) +
    'model=' + MODEL_NAME + chr(10) +
    'architecture=YOLO26s+SCConv' + chr(10) +
    'c3k2_scconv_expected_count=2' + chr(10) +
    'sampler=WeightedRandomSampler(train_only)' + chr(10),
    encoding='utf-8',
)

e7_best_model, e7_checkpoint_load_mode = load_custom_scconv_checkpoint(BEST_MODEL_PATH)
print('E7 checkpoint load mode:', e7_checkpoint_load_mode)
print('C3k2_SCConv IN E7 BEST:', count_c3k2_scconv(e7_best_model.model))
assert count_c3k2_scconv(e7_best_model.model) == 2
assert BEST_MODEL_PATH.exists()

## 5. Validation, metrics, confusion matrix, and curves

Only the unchanged validation split is evaluated after training. Ultralytics writes the requested plots under the validation run directory.

In [ ]:
def safe_f1(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def extract_metrics(result, label):
    summary = {
        'Precision': float(result.box.mp), 'Recall': float(result.box.mr),
        'mAP50': float(result.box.map50), 'mAP50-95': float(result.box.map),
    }
    summary['F1'] = safe_f1(summary['Precision'], summary['Recall'])
    summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision, recall, ap50, ap5095 = result.box.class_result(class_id)
        values = {'Precision': float(precision), 'Recall': float(recall), 'AP50': float(ap50), 'AP50-95': float(ap5095)}
        values['F1'] = safe_f1(values['Precision'], values['Recall'])
        summary['by_class'][class_name] = values
    print(label, summary)
    return summary

e7_validation_result = e7_best_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=True)
e7_validation_summary = extract_metrics(e7_validation_result, 'E7 validation')
test_metrics = None
test_summary = None
assert test_metrics is None and test_summary is None

def confusion_array(result):
    matrix = getattr(getattr(result, 'confusion_matrix', None), 'matrix', None)
    matrix = None if matrix is None else np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix is not None and matrix.shape == expected_shape else None

def confusion_error_counts(result):
    matrix = confusion_array(result)
    if matrix is None:
        return None
    labels = CLASS_NAMES + ['background']
    index = {label: i for i, label in enumerate(labels)}
    nohelmet = index['rider_without_helmet']
    return {
        'TP nohelmet': float(matrix[nohelmet, nohelmet]),
        'rider_without_helmet -> rider_with_helmet': float(matrix[index['rider_with_helmet'], nohelmet]),
        'rider_without_helmet -> motorcycle': float(matrix[index['motorcycle'], nohelmet]),
        'rider_without_helmet -> background': float(matrix[index['background'], nohelmet]),
        'background -> rider_without_helmet': float(matrix[nohelmet, index['background']]),
    }

def nohelmet_normalized_diagonal(result):
    matrix = confusion_array(result)
    if matrix is None:
        return None
    nohelmet_index = CLASS_NAMES.index('rider_without_helmet')
    denominator = matrix[:, nohelmet_index].sum()
    return float(matrix[nohelmet_index, nohelmet_index] / denominator) if denominator else 0.0

e7_confusion_errors = confusion_error_counts(e7_validation_result)
e7_nohelmet_diagonal = nohelmet_normalized_diagonal(e7_validation_result)
print('E7 no-helmet normalized diagonal:', e7_nohelmet_diagonal)
if e7_confusion_errors is not None:
    display(pd.DataFrame({'Metric': list(e7_confusion_errors), 'Count': list(e7_confusion_errors.values())}))

plot_specs = [
    ('results.png', 'Training results'), ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box PR curve'), ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'), ('BoxR_curve.png', 'Box Recall curve'),
]
for filename, title in plot_specs:
    paths = list(EXPERIMENT_DIR.rglob(filename)) if EXPERIMENT_DIR.exists() else []
    if paths:
        path = max(paths, key=lambda item: item.stat().st_mtime)
        plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
        plt.imshow(Image.open(path))
        plt.axis('off')
        plt.title(title)
        plt.show()

## 6. Same-validation E5 vs E7 comparison

This is the requested comparison exception: the E5 best checkpoint is evaluated on the same `val` split as E7. Neither model is evaluated on `test`.

In [ ]:
e5_validation_result = e5_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False)
e5_validation_summary = extract_metrics(e5_validation_result, 'E5 validation')

def comparison_values(summary):
    nohelmet = summary['by_class']['rider_without_helmet']
    return {
        'Overall Precision': summary['Precision'], 'Overall Recall': summary['Recall'],
        'Overall F1': summary['F1'], 'mAP50': summary['mAP50'], 'mAP50-95': summary['mAP50-95'],
        'NoHelmet Precision': nohelmet['Precision'], 'NoHelmet Recall': nohelmet['Recall'],
        'NoHelmet F1': nohelmet['F1'], 'NoHelmet AP50': nohelmet['AP50'],
        'NoHelmet AP50-95': nohelmet['AP50-95'],
    }

e5_values = comparison_values(e5_validation_summary)
e7_values = comparison_values(e7_validation_summary)
comparison = pd.DataFrame({
    'Metric': list(e5_values), 'E5 SCConv': list(e5_values.values()),
    'E7 SCConv + HardMining': [e7_values[key] for key in e5_values],
})
comparison['Delta'] = comparison['E7 SCConv + HardMining'] - comparison['E5 SCConv']
display(comparison)

e5_errors = confusion_error_counts(e5_validation_result)
e7_errors = confusion_error_counts(e7_validation_result)
if e5_errors is not None and e7_errors is not None:
    error_comparison = pd.DataFrame({
        'Metric': list(e5_errors), 'E5 SCConv': list(e5_errors.values()),
        'E7 SCConv + HardMining': [e7_errors[key] for key in e5_errors],
    })
    error_comparison['Delta'] = error_comparison['E7 SCConv + HardMining'] - error_comparison['E5 SCConv']
    display(error_comparison)

comparison_with_diagonal = pd.DataFrame([{
    'Metric': 'NoHelmet normalized confusion diagonal',
    'E5 SCConv': nohelmet_normalized_diagonal(e5_validation_result),
    'E7 SCConv + HardMining': e7_nohelmet_diagonal,
}])
comparison_with_diagonal['Delta'] = comparison_with_diagonal['E7 SCConv + HardMining'] - comparison_with_diagonal['E5 SCConv']
display(comparison_with_diagonal)
e5_nohelmet_diagonal = nohelmet_normalized_diagonal(e5_validation_result)

## 7. Final audit

The audit is deliberately strict: it checks the pinned version, fresh E7 initialization, exact architecture count, train-only mining, standard validation loader, no forbidden modules, and the unchanged E5 training protocol.

In [ ]:
assert ultralytics.__version__ == '8.4.155'
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG == {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15, 'pretrained': True,
}
assert VALIDATION_SPLIT == 'val' and MINING_SPLIT == 'train' and TEST_USED is False
assert c3k2_scconv_before_train == c3k2_scconv_inside_trainer == count_c3k2_scconv(e7_best_model.model) == 2
assert count_c3k2_scconv(e5_model.model) == 2
assert scconv_grad_ok and count_scconv(e7_best_model.model) > 0
assert e7_model is not e5_model and BEST_MODEL_PATH != E5_BEST_MODEL_PATH
assert HARD_REPORT_PATH.exists() and len(hard_samples) == len(TRAIN_IMAGE_PATHS)
assert isinstance(trainer.train_loader.sampler, WeightedRandomSampler)
assert trainer.train_loader.sampler.replacement
assert trainer.train_loader.sampler.num_samples == len(trainer.train_loader.dataset)
assert not isinstance(trainer.test_loader.sampler, WeightedRandomSampler)
forbidden = {'P2', 'iAFF', 'MultiSEAM', 'DySample', 'SEAM', 'CBAM', 'ECA', 'AFPN', 'ADown'}
e7_module_names = {item.__class__.__name__ for item in e7_best_model.model.modules()}
assert not e7_module_names.intersection(forbidden)
assert EXPERIMENT_VERSION_FILE.exists()
print('Final audit passed.')
print('Ultralytics:', ultralytics.__version__)
print('C3k2_SCConv before/train/best:', c3k2_scconv_before_train, c3k2_scconv_inside_trainer, count_c3k2_scconv(e7_best_model.model))
print('Train sampler:', type(trainer.train_loader.sampler).__name__, 'replacement=', trainer.train_loader.sampler.replacement, 'num_samples=', trainer.train_loader.sampler.num_samples)
print('Validation sampler:', type(trainer.test_loader.sampler).__name__)
print('Mining split:', MINING_SPLIT, '| Validation split:', VALIDATION_SPLIT, '| Test used:', TEST_USED)
print('Output:', BEST_MODEL_PATH)
print('NoHelmet normalized diagonal E5 -> E7:', e5_nohelmet_diagonal, '->', e7_nohelmet_diagonal)

Fixed only:
1. weighted sampler path-order alignment
2. E5 checkpoint loading validation

Architecture, mining logic and training protocol unchanged.